In [ ]:
#testsing 15 task of bigBench in phi3
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils.bigbench_utils import run_task_eval

model_name = "microsoft/Phi-3-mini-128k-instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="cpu", torch_dtype="auto", attn_implementation="eager"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

def ask_phi3(prompt):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=50, do_sample=False, pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

RESULTS_FILE = "../results_phi3_bigbench.csv"

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
def run_and_save(task_name, ask_fn, model_label, n=20):
    print(f"Running {task_name}...")
    result = run_task_eval(task_name, ask_fn, n=n)
    result["model"] = model_label

    df_new = pd.DataFrame([result])
    if os.path.exists(RESULTS_FILE):
        df_existing = pd.read_csv(RESULTS_FILE)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new
    df_combined.to_csv(RESULTS_FILE, index=False)

    print(f"  accuracy={result['accuracy']:.1%}  avg_time={result['avg_time_per_example_sec']}s  (saved)")
    return df_combined

In [5]:
run_and_save("causal_judgment", ask_phi3, "Phi-3-mini")

Running causal_judgment...


You are not running the flash-attention implementation, expect numerical differences.


  accuracy=65.0%  avg_time=65.94s  (saved)


,task,n_examples,accuracy,total_time_sec,avg_time_per_example_sec,model
0,causal_judgment,20,0.65,1318.79,65.94,Phi-3-mini


In [6]:
run_and_save("key_value_maps", ask_phi3, "Phi-3-mini")

Running key_value_maps...


key_value_maps/validation-00000-of-00001(…):   0%|          | 0.00/9.02k [00:00<?, ?B/s]

c:\Users\shres_sm1uxfb\Desktop\llm-models\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shres_sm1uxfb\.cache\huggingface\hub\datasets--tasksource--bigbench. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


  accuracy=65.0%  avg_time=74.94s  (saved)


,task,n_examples,accuracy,total_time_sec,avg_time_per_example_sec,model
0,causal_judgment,20,0.65,1318.79,65.94,Phi-3-mini
1,key_value_maps,20,0.65,1498.89,74.94,Phi-3-mini


In [7]:
run_and_save("figure_of_speech_detection", ask_phi3, "Phi-3-mini")
run_and_save("date_understanding", ask_phi3, "Phi-3-mini")

Running figure_of_speech_detection...


figure_of_speech_detection/validation-00(…):   0%|          | 0.00/4.90k [00:00<?, ?B/s]

  accuracy=68.8%  avg_time=31.48s  (saved)
Running date_understanding...
  accuracy=65.0%  avg_time=38.07s  (saved)


,task,n_examples,accuracy,total_time_sec,avg_time_per_example_sec,model
0,causal_judgment,20,0.6500,1318.79,65.94,Phi-3-mini
1,key_value_maps,20,0.6500,1498.89,74.94,Phi-3-mini
2,figure_of_speech_detection,16,0.6875,503.76,31.48,Phi-3-mini
3,date_understanding,20,0.6500,761.42,38.07,Phi-3-mini


In [ ]:
#Testing 15 bigBench tasks in LLaMA-3-8B
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import pandas as pd
from llama_cpp import Llama
from utils.bigbench_utils import run_task_eval

llm = Llama.from_pretrained(
    repo_id="bartowski/Meta-Llama-3-8B-Instruct-GGUF",
    filename="Meta-Llama-3-8B-Instruct-Q4_K_M.gguf",
    n_ctx=4096, n_gpu_layers=0, verbose=False,
)

def ask_llama3(prompt):
    output = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}], max_tokens=50, temperature=0.0
    )
    return output["choices"][0]["message"]["content"]

RESULTS_FILE = "../results_llama3_bigbench.csv"

In [9]:
def run_and_save(task_name, ask_fn, model_label, n=20):
    print(f"Running {task_name}...")
    result = run_task_eval(task_name, ask_fn, n=n)
    result["model"] = model_label

    df_new = pd.DataFrame([result])
    if os.path.exists(RESULTS_FILE):
        df_existing = pd.read_csv(RESULTS_FILE)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new
    df_combined.to_csv(RESULTS_FILE, index=False)

    print(f"  accuracy={result['accuracy']:.1%}  avg_time={result['avg_time_per_example_sec']}s  (saved)")
    return df_combined

In [10]:
run_and_save("fantasy_reasoning", ask_llama3, "Llama-3-8B")

run_and_save("moral_permissibility", ask_llama3, "Llama-3-8B")

run_and_save("hinglish_toxicity", ask_llama3, "Llama-3-8B")

run_and_save("mathematical_induction", ask_llama3, "Llama-3-8B")

run_and_save("auto_categorization", ask_llama3, "Llama-3-8B")

run_and_save("implicit_relations", ask_llama3, "Llama-3-8B")

run_and_save("medical_questions_russian", ask_llama3, "Llama-3-8B")

run_and_save("repeat_copy_logic", ask_llama3, "Llama-3-8B")

run_and_save("analytic_entailment", ask_llama3, "Llama-3-8B")

run_and_save("anachronisms", ask_llama3, "Llama-3-8B")

run_and_save("analogical_similarity", ask_llama3, "Llama-3-8B")

run_and_save("causal_judgment", ask_llama3, "Llama-3-8B")

run_and_save("key_value_maps", ask_llama3, "Llama-3-8B")

run_and_save("figure_of_speech_detection", ask_llama3, "Llama-3-8B")

run_and_save("date_understanding", ask_llama3, "Llama-3-8B")

Running fantasy_reasoning...
  accuracy=60.0%  avg_time=11.87s  (saved)
Running moral_permissibility...
  accuracy=60.0%  avg_time=12.98s  (saved)
Running hinglish_toxicity...
  accuracy=55.0%  avg_time=12.57s  (saved)
Running mathematical_induction...
  accuracy=68.8%  avg_time=8.5s  (saved)
Running auto_categorization...
  accuracy=0.0%  avg_time=7.12s  (saved)
Running implicit_relations...
  accuracy=64.7%  avg_time=26.35s  (saved)
Running medical_questions_russian...
  accuracy=85.0%  avg_time=24.55s  (saved)
Running repeat_copy_logic...
  accuracy=0.0%  avg_time=7.73s  (saved)
Running analytic_entailment...
  accuracy=56.2%  avg_time=5.52s  (saved)
Running anachronisms...
  accuracy=45.0%  avg_time=6.28s  (saved)
Running analogical_similarity...
  accuracy=15.0%  avg_time=15.92s  (saved)
Running causal_judgment...
  accuracy=50.0%  avg_time=22.93s  (saved)
Running key_value_maps...
  accuracy=45.0%  avg_time=8.32s  (saved)
Running figure_of_speech_detection...
  accuracy=68.8%  av

,task,n_examples,accuracy,total_time_sec,avg_time_per_example_sec,model
0,fantasy_reasoning,20,0.600000,237.35,11.87,Llama-3-8B
1,moral_permissibility,20,0.600000,259.65,12.98,Llama-3-8B
2,hinglish_toxicity,20,0.550000,251.46,12.57,Llama-3-8B
3,mathematical_induction,16,0.687500,135.98,8.50,Llama-3-8B
4,auto_categorization,20,0.000000,142.43,7.12,Llama-3-8B
5,implicit_relations,17,0.647059,447.94,26.35,Llama-3-8B
6,medical_questions_russian,20,0.850000,491.09,24.55,Llama-3-8B
7,repeat_copy_logic,16,0.000000,123.61,7.73,Llama-3-8B
8,analytic_entailment,16,0.562500,88.25,5.52,Llama-3-8B
9,anachronisms,20,0.450000,125.52,6.28,Llama-3-8B


In [15]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

os.environ["HF_TOKEN"] = "your_current_valid_token_here"

import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils.bigbench_utils import run_task_eval

model_name = "meta-llama/Llama-3.2-1B"
print("Loading Llama-3.2-1B...")
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="cpu", torch_dtype="auto", attn_implementation="eager"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

def ask_llama32(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    if inputs["input_ids"][0][-1].item() == tokenizer.eos_token_id:
        inputs["input_ids"] = inputs["input_ids"][:, :-1]
        inputs["attention_mask"] = inputs["attention_mask"][:, :-1]
    inputs = inputs.to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=50, min_new_tokens=1, do_sample=False, pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

# New filename — keeps the old (broken) results separate rather than overwriting/mixing
RESULTS_FILE = "../results_llama32_bigbench_v2.csv"

Loading Llama-3.2-1B...


In [16]:
def run_and_save(task_name, ask_fn, model_label, n=20):
    print(f"Running {task_name}...")
    result = run_task_eval(task_name, ask_fn, n=n)
    result["model"] = model_label

    df_new = pd.DataFrame([result])
    if os.path.exists(RESULTS_FILE):
        df_existing = pd.read_csv(RESULTS_FILE)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new
    df_combined.to_csv(RESULTS_FILE, index=False)

    print(f"  accuracy={result['accuracy']:.1%}  avg_time={result['avg_time_per_example_sec']}s  (saved)")
    return df_combined

In [ ]:

run_and_save("fantasy_reasoning", ask_llama32, "Llama-3.2-1B")

run_and_save("moral_permissibility", ask_llama32, "Llama-3.2-1B")

run_and_save("hinglish_toxicity", ask_llama32, "Llama-3.2-1B")

run_and_save("mathematical_induction", ask_llama32, "Llama-3.2-1B")

run_and_save("auto_categorization", ask_llama32, "Llama-3.2-1B")

run_and_save("implicit_relations", ask_llama32, "Llama-3.2-1B")

run_and_save("medical_questions_russian", ask_llama32, "Llama-3.2-1B")

run_and_save("repeat_copy_logic", ask_llama32, "Llama-3.2-1B")

run_and_save("analytic_entailment", ask_llama32, "Llama-3.2-1B")

run_and_save("anachronisms", ask_llama32, "Llama-3.2-1B")

run_and_save("analogical_similarity", ask_llama32, "Llama-3.2-1B")

run_and_save("causal_judgment", ask_llama32, "Llama-3.2-1B")

run_and_save("key_value_maps", ask_llama32, "Llama-3.2-1B")

run_and_save("figure_of_speech_detection", ask_llama32, "Llama-3.2-1B")

run_and_save("date_understanding", ask_llama32, "Llama-3.2-1B")

Running fantasy_reasoning...


c:\Users\shres_sm1uxfb\Desktop\llm-models\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\shres_sm1uxfb\Desktop\llm-models\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  accuracy=0.0%  avg_time=8.8s  (saved)
Running moral_permissibility...


c:\Users\shres_sm1uxfb\Desktop\llm-models\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\shres_sm1uxfb\Desktop\llm-models\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  accuracy=50.0%  avg_time=8.74s  (saved)
Running hinglish_toxicity...


...

  accuracy=0.0%  avg_time=5.9s  (saved)
Running date_understanding...


c:\Users\shres_sm1uxfb\Desktop\llm-models\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\shres_sm1uxfb\Desktop\llm-models\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  accuracy=0.0%  avg_time=6.1s  (saved)


,task,n_examples,accuracy,total_time_sec,avg_time_per_example_sec,model
0,fantasy_reasoning,20,0.000,176.06,8.80,Llama-3.2-1B
1,moral_permissibility,20,0.500,174.85,8.74,Llama-3.2-1B
2,hinglish_toxicity,20,0.800,171.54,8.58,Llama-3.2-1B
3,mathematical_induction,16,0.375,85.55,5.35,Llama-3.2-1B
4,auto_categorization,20,0.000,159.22,7.96,Llama-3.2-1B
5,implicit_relations,17,0.000,272.70,16.04,Llama-3.2-1B
6,medical_questions_russian,20,0.000,333.84,16.69,Llama-3.2-1B
7,repeat_copy_logic,16,0.000,123.23,7.70,Llama-3.2-1B
8,analytic_entailment,16,0.000,92.92,5.81,Llama-3.2-1B
9,anachronisms,20,0.550,166.48,8.32,Llama-3.2-1B


: 

In [8]:
#testing with NLLB-200-1.3B
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from utils.bigbench_utils import run_task_eval

MODEL_NAME = "facebook/nllb-200-1.3B"
print("Loading NLLB...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def ask_nllb(prompt):
    tokenizer.src_lang = "eng_Latn"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    generated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"),
        max_length=50,
    )
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

RESULTS_FILE = "../results_nllb_bigbench.csv"

Loading NLLB...


c:\Users\shres_sm1uxfb\Desktop\llm-models\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
def run_and_save(task_name, ask_fn, model_label, n=20):
    print(f"Running {task_name}...")
    result = run_task_eval(task_name, ask_fn, n=n)
    result["model"] = model_label

    df_new = pd.DataFrame([result])
    if os.path.exists(RESULTS_FILE):
        df_existing = pd.read_csv(RESULTS_FILE)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new
    df_combined.to_csv(RESULTS_FILE, index=False)

    print(f"  accuracy={result['accuracy']:.1%}  avg_time={result['avg_time_per_example_sec']}s  (saved)")
    return df_combined

In [10]:
run_and_save("fantasy_reasoning", ask_nllb, "NLLB-200")

run_and_save("moral_permissibility", ask_nllb, "NLLB-200")

run_and_save("hinglish_toxicity", ask_nllb, "NLLB-200")

run_and_save("mathematical_induction", ask_nllb, "NLLB-200")

run_and_save("auto_categorization", ask_nllb, "NLLB-200")

run_and_save("implicit_relations", ask_nllb, "NLLB-200")

run_and_save("medical_questions_russian", ask_nllb, "NLLB-200")

run_and_save("repeat_copy_logic", ask_nllb, "NLLB-200")

run_and_save("analytic_entailment", ask_nllb, "NLLB-200")

run_and_save("anachronisms", ask_nllb, "NLLB-200")

run_and_save("analogical_similarity", ask_nllb, "NLLB-200")

run_and_save("causal_judgment", ask_nllb, "NLLB-200")

run_and_save("key_value_maps", ask_nllb, "NLLB-200")

run_and_save("figure_of_speech_detection", ask_nllb, "NLLB-200")

run_and_save("date_understanding", ask_nllb, "NLLB-200")

Running fantasy_reasoning...
  accuracy=0.0%  avg_time=56.87s  (saved)
Running moral_permissibility...
  accuracy=10.0%  avg_time=58.02s  (saved)
Running hinglish_toxicity...
  accuracy=60.0%  avg_time=55.53s  (saved)
Running mathematical_induction...
  accuracy=12.5%  avg_time=53.5s  (saved)
Running auto_categorization...
  accuracy=0.0%  avg_time=42.49s  (saved)
Running implicit_relations...
  accuracy=0.0%  avg_time=64.55s  (saved)
Running medical_questions_russian...
  accuracy=5.0%  avg_time=61.38s  (saved)
Running repeat_copy_logic...
  accuracy=0.0%  avg_time=45.51s  (saved)
Running analytic_entailment...
  accuracy=50.0%  avg_time=52.57s  (saved)
Running anachronisms...
  accuracy=95.0%  avg_time=51.28s  (saved)
Running analogical_similarity...
  accuracy=0.0%  avg_time=76.81s  (saved)
Running causal_judgment...
  accuracy=0.0%  avg_time=64.25s  (saved)
Running key_value_maps...
  accuracy=55.0%  avg_time=66.09s  (saved)
Running figure_of_speech_detection...
  accuracy=18.8%  a

,task,n_examples,accuracy,total_time_sec,avg_time_per_example_sec,model
0,fantasy_reasoning,20,0.0000,1137.35,56.87,NLLB-200
1,moral_permissibility,20,0.1000,1160.41,58.02,NLLB-200
2,hinglish_toxicity,20,0.6000,1110.58,55.53,NLLB-200
3,mathematical_induction,16,0.1250,855.96,53.50,NLLB-200
4,auto_categorization,20,0.0000,849.82,42.49,NLLB-200
5,implicit_relations,17,0.0000,1097.43,64.55,NLLB-200
6,medical_questions_russian,20,0.0500,1227.63,61.38,NLLB-200
7,repeat_copy_logic,16,0.0000,728.16,45.51,NLLB-200
8,analytic_entailment,16,0.5000,841.05,52.57,NLLB-200
9,anachronisms,20,0.9500,1025.52,51.28,NLLB-200


In [11]:
from utils.bigbench_utils import load_task, build_prompt, get_correct_answers

examples = load_task("anachronisms", n=3)
for ex in examples:
    prompt = build_prompt(ex)
    response = ask_nllb(prompt)
    print("PROMPT (first 200 chars):", prompt[:200])
    print("CORRECT ANSWER:", get_correct_answers(ex))
    print("NLLB RESPONSE:", repr(response))
    print("---")

PROMPT (first 200 chars): Chloe Zhao caught up on her sleep in her favorite nap pod, the EnergyPod.
Does the preceding sentence contain non-contemporaneous (anachronistic) elements?

Choices:
- Yes
- No

Answer with the exact 
CORRECT ANSWER: {'No'}
NLLB RESPONSE: 'Chloe Zhao caught up on her sleep in her favorite nap pod, the EnergyPod. Does the preceding sentence contain non-contemporary (anachronistic) elements? Choices: - Yes - No Answer with the exact text of'
---
PROMPT (first 200 chars): Taiwan was excluded from the WTO meeting about Covid19.
Does the preceding sentence contain non-contemporaneous (anachronistic) elements?

Choices:
- Yes
- No

Answer with the exact text of the correc
CORRECT ANSWER: {'No'}
NLLB RESPONSE: 'Taiwan was excluded from the WTO meeting about Covid19.[citation needed] Does the preceding sentence contain non-contemporary (anachronistic) elements? Choices: - Yes - No Answer with the exact text of the correct'
---
PROMPT (first 200 chars): Mitch McConnell

In [ ]:
from utils.bigbench_utils import load_task, build_prompt, get_correct_answers

examples = load_task("causal", n=3)
for ex in examples:
    prompt = build_prompt(ex)
    response = ask_llama32(prompt)
    print("Correct answer(s):", get_correct_answers(ex))
    print("Model response:", repr(response))
    print("---")

Correct answer(s): {'03/13/2002'}
Model response: ''
---
Correct answer(s): {'12/30/2014'}
Model response: ''
---
Correct answer(s): {'06/19/2019'}
Model response: ''
---


In [13]:
example = load_task("date_understanding", n=1)[0]
prompt = build_prompt(example)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs, max_new_tokens=50, do_sample=False, pad_token_id=tokenizer.eos_token_id
)

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
print("Number of new tokens generated:", len(new_tokens))
print("Raw token IDs:", new_tokens.tolist())
print("Decoded WITHOUT skipping special tokens:", repr(tokenizer.decode(new_tokens, skip_special_tokens=False)))

Number of new tokens generated: 0
Raw token IDs: []
Decoded WITHOUT skipping special tokens: ''


In [14]:
example = load_task("date_understanding", n=1)[0]
prompt = build_prompt(example)

inputs = tokenizer(prompt, return_tensors="pt")
print("Last 5 input token IDs:", inputs["input_ids"][0][-5:].tolist())
print("EOS token ID:", tokenizer.eos_token_id)
print("Decoded last 10 tokens of input:", repr(tokenizer.decode(inputs["input_ids"][0][-10:])))

Last 5 input token IDs: [70131, 86972, 12468, 248075, 2]
EOS token ID: 2
Decoded last 10 tokens of input: 'the exact text of the correct choice only.</s>'


In [4]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

from utils.bigbench_utils import load_task

BBL_TASKS = [
    "auto_debugging", "bbq_lite_json", "code_line_description", "conceptual_combinations",
    "conlang_translation", "emoji_movie", "formal_fallacies_syllogisms_negation", "hindu_knowledge",
    "known_unknowns", "language_identification", "linguistics_puzzles", "logic_grid_puzzle",
    "logical_deduction", "misconceptions_russian", "novel_concepts", "operators",
    "parsinlu_reading_comprehension", "play_dialog_same_or_different", "repeat_copy_logic",
    "strange_stories", "strategyqa", "symbol_interpretation", "vitaminc_fact_verification", "winowhy",
]

for task in BBL_TASKS:
    try:
        ex = load_task(task, n=1)[0]
        print(f"{task:40s} n_choices={len(ex['multiple_choice_targets']):2d}")
    except Exception as e:
        print(f"{task:40s} ERROR: {str(e)[:60]}")

c:\Users\shres_sm1uxfb\Desktop\llm-models\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shres_sm1uxfb\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


auto_debugging/validation-00000-of-00001(…): reconstructing file:   0%|          |  0.00B / 4.48kB            

auto_debugging/validation-00000-of-00001(…): downloading bytes:           |  0.00B            

c:\Users\shres_sm1uxfb\Desktop\llm-models\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shres_sm1uxfb\.cache\huggingface\hub\datasets--tasksource--bigbench. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


auto_debugging                           n_choices= 0


...

vitaminc_fact_verification/validation-00(…): downloading bytes:           |  0.00B            

vitaminc_fact_verification               n_choices= 3


winowhy/validation-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B / 58.6kB            

winowhy/validation-00000-of-00001.parque(…): downloading bytes:           |  0.00B            

winowhy                                  n_choices= 2


In [8]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

from utils.bigbench_utils import load_task, build_prompt, get_correct_answers

ex = load_task("repeat_copy_logic", n=1)[0]
print("Prompt:", build_prompt(ex))
print("Correct answers:", get_correct_answers(ex))

Prompt: repeat with logic:

Q: Say the days of the week but only the weekend days, two times
A:

Choices:


Answer with the exact text of the correct choice only.
Correct answers: set()


In [9]:
def run_and_save(task_name, ask_fn, model_label, n=20):
    print(f"Running {task_name}...")
    result = run_task_eval(task_name, ask_fn, n=n)
    result["model"] = model_label

    df_new = pd.DataFrame([result])
    if os.path.exists(RESULTS_FILE):
        df_existing = pd.read_csv(RESULTS_FILE)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new
    df_combined.to_csv(RESULTS_FILE, index=False)

    print(f"  accuracy={result['accuracy']:.1%}  avg_time={result['avg_time_per_example_sec']}s  (saved)")
    return df_combined

In [12]:
import inspect
from utils import bigbench_utils
print(inspect.getsource(bigbench_utils.build_prompt))
print(inspect.getsource(bigbench_utils.get_correct_answers))

def build_prompt(example):
    choices = example["multiple_choice_targets"]
    if len(choices) == 0:
        # Generative task — no fixed choices to offer.
        return f"{example['inputs']}\n\nAnswer:"
    choice_text = "\n".join(f"- {c}" for c in choices)
    return (
        f"{example['inputs']}\n\n"
        f"Choices:\n{choice_text}\n\n"
        f"Answer with the exact text of the correct choice only."
    )

def build_prompt(example):
    choices = example["multiple_choice_targets"]
    if len(choices) == 0:
        # Generative task — no fixed choices to offer.
        return f"{example['inputs']}\n\nAnswer:"
    choice_text = "\n".join(f"- {c}" for c in choices)
    return (
        f"{example['inputs']}\n\n"
        f"Choices:\n{choice_text}\n\n"
        f"Answer with the exact text of the correct choice only."
    )



In [24]:
with open("../utils/bigbench_utils.py", "r") as f:
    content = f.read()
start = content.find("def get_correct_answers")
end = content.find("\ndef ", start + 1)
print(content[start:end])

def get_correct_answers(example):
    targets = example["multiple_choice_targets"]
    scores = example["multiple_choice_scores"]
    if len(targets) == 0:
        # Generative task â€” the correct answer(s) live in `targets`, not the choice list.
        return set(example["targets"])
    return {t for t, s in zip(targets, scores) if s == 1}



In [28]:
ex = load_task("repeat_copy_logic", n=1)[0]

mc_targets = ex["multiple_choice_targets"]
print("multiple_choice_targets:", mc_targets)
print("len:", len(mc_targets))
print("len == 0?:", len(mc_targets) == 0)

multiple_choice_targets: []
len: 0
len == 0?: True


In [30]:
from utils.bigbench_utils import load_task, build_prompt, get_correct_answers, score_response

In [31]:
print(bigbench_utils.get_correct_answers(ex))
print(get_correct_answers(ex))

set()
set()


In [34]:
import importlib
from utils import bigbench_utils
importlib.reload(bigbench_utils)

print(bigbench_utils.get_correct_answers(ex))

{'Saturday Sunday Saturday Sunday'}


In [35]:
import sys
for p in sys.path:
    print(p)

C:\Users\shres_sm1uxfb\AppData\Local\Programs\Python\Python312\python312.zip
C:\Users\shres_sm1uxfb\AppData\Local\Programs\Python\Python312\DLLs
C:\Users\shres_sm1uxfb\AppData\Local\Programs\Python\Python312\Lib
C:\Users\shres_sm1uxfb\AppData\Local\Programs\Python\Python312
c:\Users\shres_sm1uxfb\Desktop\llm-models\.venv

c:\Users\shres_sm1uxfb\Desktop\llm-models\.venv\Lib\site-packages
c:\Users\shres_sm1uxfb\Desktop\llm-models
c:\Users\shres_sm1uxfb\Desktop\llm-models
c:\Users\shres_sm1uxfb\Desktop\llm-models
c:\Users\shres_sm1uxfb\Desktop\llm-models
c:\Users\shres_sm1uxfb\Desktop\llm-models


In [36]:
import shutil
shutil.rmtree("../utils/__pycache__", ignore_errors=True)

In [1]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

from utils.bigbench_utils import load_task, build_prompt, get_correct_answers, score_response

ex = load_task("repeat_copy_logic", n=1)[0]
print("Prompt:", build_prompt(ex))
print("Correct answers:", get_correct_answers(ex))

Prompt: repeat with logic:

Q: Say the days of the week but only the weekend days, two times
A:

Answer:
Correct answers: {'Saturday Sunday Saturday Sunday'}


In [1]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils.bigbench_utils import run_task_eval

model_name = "microsoft/Phi-3-mini-128k-instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="cpu", torch_dtype="auto", attn_implementation="eager"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

def ask_phi3(prompt):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=50, do_sample=False, pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

RESULTS_FILE = "../results_phi3_bigbench.csv"

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading weights:   0%|          | 0/195 [00:19<?, ?it/s]

In [2]:
def run_and_save(task_name, ask_fn, model_label, n=20):
    print(f"Running {task_name}...")
    result = run_task_eval(task_name, ask_fn, n=n)
    result["model"] = model_label

    df_new = pd.DataFrame([result])
    if os.path.exists(RESULTS_FILE):
        df_existing = pd.read_csv(RESULTS_FILE)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new
    df_combined.to_csv(RESULTS_FILE, index=False)

    print(f"  accuracy={result['accuracy']:.1%}  avg_time={result['avg_time_per_example_sec']}s  (saved)")
    return df_combined

In [3]:
run_and_save("repeat_copy_logic", ask_phi3, "Phi-3-mini")

Running repeat_copy_logic...
  accuracy=0.0%  avg_time=30.16s  (saved)


,task,n_examples,accuracy,total_time_sec,avg_time_per_example_sec,model
0,repeat_copy_logic,16,0.0,482.55,30.16,Phi-3-mini


Testing of BigBench Lite dataset on Phi3, Ollama, Qwen and Granite

In [1]:
#Phi3 mini
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils.bigbench_utils import run_task_eval

model_name = "microsoft/Phi-3-mini-128k-instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="cpu", torch_dtype="auto", attn_implementation="eager"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

def ask_phi3(prompt):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=50, do_sample=False, pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

def run_and_save(task_name, ask_fn, model_label, n=20):
    print(f"Running {task_name}...")
    result = run_task_eval(task_name, ask_fn, n=n)
    result["model"] = model_label
    df_new = pd.DataFrame([result])
    if os.path.exists(RESULTS_FILE):
        df_existing = pd.read_csv(RESULTS_FILE)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new
    df_combined.to_csv(RESULTS_FILE, index=False)
    print(f"  accuracy={result['accuracy']:.1%}  avg_time={result['avg_time_per_example_sec']}s  (saved)")
    return df_combined

RESULTS_FILE = "../results_phi3_bbl.csv"

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading weights:   0%|          | 0/195 [00:07<?, ?it/s]

In [ ]:

run_and_save("auto_debugging", ask_phi3, "Phi-3-mini")

run_and_save("bbq_lite_json", ask_phi3, "Phi-3-mini")

run_and_save("code_line_description", ask_phi3, "Phi-3-mini")

run_and_save("conceptual_combinations", ask_phi3, "Phi-3-mini")

run_and_save("conlang_translation", ask_phi3, "Phi-3-mini")

run_and_save("emoji_movie", ask_phi3, "Phi-3-mini")

run_and_save("formal_fallacies_syllogisms_negation", ask_phi3, "Phi-3-mini")

run_and_save("hindu_knowledge", ask_phi3, "Phi-3-mini")

run_and_save("known_unknowns", ask_phi3, "Phi-3-mini")

run_and_save("language_identification", ask_phi3, "Phi-3-mini")

run_and_save("linguistics_puzzles", ask_phi3, "Phi-3-mini")

run_and_save("logic_grid_puzzle", ask_phi3, "Phi-3-mini")

run_and_save("logical_deduction", ask_phi3, "Phi-3-mini")

run_and_save("misconceptions_russian", ask_phi3, "Phi-3-mini")

run_and_save("novel_concepts", ask_phi3, "Phi-3-mini")

run_and_save("operators", ask_phi3, "Phi-3-mini")

run_and_save("parsinlu_reading_comprehension", ask_phi3, "Phi-3-mini")

run_and_save("play_dialog_same_or_different", ask_phi3, "Phi-3-mini")

run_and_save("repeat_copy_logic", ask_phi3, "Phi-3-mini")

run_and_save("strange_stories", ask_phi3, "Phi-3-mini")

run_and_save("strategyqa", ask_phi3, "Phi-3-mini")

run_and_save("symbol_interpretation", ask_phi3, "Phi-3-mini")

run_and_save("vitaminc_fact_verification", ask_phi3, "Phi-3-mini")

run_and_save("winowhy", ask_phi3, "Phi-3-mini")
#428m 10.3s

Running auto_debugging...
  accuracy=75.0%  avg_time=42.4s  (saved)
Running bbq_lite_json...
  accuracy=75.0%  avg_time=27.2s  (saved)
Running code_line_description...
  accuracy=93.8%  avg_time=39.08s  (saved)
Running conceptual_combinations...
  accuracy=84.2%  avg_time=46.26s  (saved)
Running conlang_translation...
  accuracy=15.0%  avg_time=125.66s  (saved)
Running emoji_movie...
  accuracy=70.0%  avg_time=33.26s  (saved)
Running formal_fallacies_syllogisms_negation...
  accuracy=85.0%  avg_time=45.82s  (saved)
Running hindu_knowledge...
  accuracy=65.0%  avg_time=23.39s  (saved)
Running known_unknowns...
  accuracy=93.8%  avg_time=21.83s  (saved)
Running language_identification...
  accuracy=15.0%  avg_time=58.98s  (saved)
Running linguistics_puzzles...
  accuracy=0.0%  avg_time=79.79s  (saved)
Running logic_grid_puzzle...
  accuracy=50.0%  avg_time=76.18s  (saved)
Running logical_deduction...
  accuracy=75.0%  avg_time=48.14s  (saved)
Running misconceptions_russian...
  accuracy=

,task,n_examples,accuracy,total_time_sec,avg_time_per_example_sec,model
0,auto_debugging,16,0.750000,678.39,42.40,Phi-3-mini
1,bbq_lite_json,20,0.750000,544.01,27.20,Phi-3-mini
2,code_line_description,16,0.937500,625.32,39.08,Phi-3-mini
3,conceptual_combinations,19,0.842105,878.97,46.26,Phi-3-mini
4,conlang_translation,20,0.150000,2513.17,125.66,Phi-3-mini
5,emoji_movie,20,0.700000,665.21,33.26,Phi-3-mini
6,formal_fallacies_syllogisms_negation,20,0.850000,916.34,45.82,Phi-3-mini
7,hindu_knowledge,20,0.650000,467.82,23.39,Phi-3-mini
8,known_unknowns,16,0.937500,349.24,21.83,Phi-3-mini
9,language_identification,20,0.150000,1179.52,58.98,Phi-3-mini


In [3]:
#llama 3b qunatized
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import pandas as pd
from llama_cpp import Llama
from utils.bigbench_utils import run_task_eval

llm = Llama.from_pretrained(
    repo_id="bartowski/Meta-Llama-3-8B-Instruct-GGUF",
    filename="Meta-Llama-3-8B-Instruct-Q4_K_M.gguf",
    n_ctx=4096, n_gpu_layers=0, verbose=False,
)

def ask_llama3(prompt):
    output = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}], max_tokens=50, temperature=0.0
    )
    return output["choices"][0]["message"]["content"]

def run_and_save(task_name, ask_fn, model_label, n=20):
    print(f"Running {task_name}...")
    result = run_task_eval(task_name, ask_fn, n=n)
    result["model"] = model_label
    df_new = pd.DataFrame([result])
    if os.path.exists(RESULTS_FILE):
        df_existing = pd.read_csv(RESULTS_FILE)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new
    df_combined.to_csv(RESULTS_FILE, index=False)
    print(f"  accuracy={result['accuracy']:.1%}  avg_time={result['avg_time_per_example_sec']}s  (saved)")
    return df_combined

RESULTS_FILE = "../results_llama3_bbl.csv"

c:\Users\shres_sm1uxfb\Desktop\llm-models\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


In [4]:

run_and_save("auto_debugging", ask_llama3, "Llama-3-8B")

run_and_save("bbq_lite_json", ask_llama3, "Llama-3-8B")

run_and_save("code_line_description", ask_llama3, "Llama-3-8B")

run_and_save("conceptual_combinations", ask_llama3, "Llama-3-8B")

run_and_save("conlang_translation", ask_llama3, "Llama-3-8B")

run_and_save("emoji_movie", ask_llama3, "Llama-3-8B")

run_and_save("formal_fallacies_syllogisms_negation", ask_llama3, "Llama-3-8B")

run_and_save("hindu_knowledge", ask_llama3, "Llama-3-8B")

run_and_save("known_unknowns", ask_llama3, "Llama-3-8B")

run_and_save("language_identification", ask_llama3, "Llama-3-8B")

run_and_save("linguistics_puzzles", ask_llama3, "Llama-3-8B")

run_and_save("logic_grid_puzzle", ask_llama3, "Llama-3-8B")

run_and_save("logical_deduction", ask_llama3, "Llama-3-8B")

run_and_save("misconceptions_russian", ask_llama3, "Llama-3-8B")

run_and_save("novel_concepts", ask_llama3, "Llama-3-8B")

run_and_save("operators", ask_llama3, "Llama-3-8B")

run_and_save("parsinlu_reading_comprehension", ask_llama3, "Llama-3-8B")

run_and_save("play_dialog_same_or_different", ask_llama3, "Llama-3-8B")

run_and_save("repeat_copy_logic", ask_llama3, "Llama-3-8B")

run_and_save("strange_stories", ask_llama3, "Llama-3-8B")

run_and_save("strategyqa", ask_llama3, "Llama-3-8B")

run_and_save("symbol_interpretation", ask_llama3, "Llama-3-8B")

run_and_save("vitaminc_fact_verification", ask_llama3, "Llama-3-8B")

run_and_save("winowhy", ask_llama3, "Llama-3-8B")

Running auto_debugging...
  accuracy=56.2%  avg_time=20.27s  (saved)
Running bbq_lite_json...
  accuracy=15.0%  avg_time=10.2s  (saved)
Running code_line_description...
  accuracy=100.0%  avg_time=15.24s  (saved)
Running conceptual_combinations...
  accuracy=84.2%  avg_time=14.78s  (saved)
Running conlang_translation...
  accuracy=10.0%  avg_time=25.44s  (saved)
Running emoji_movie...
  accuracy=90.0%  avg_time=10.12s  (saved)
Running formal_fallacies_syllogisms_negation...
  accuracy=50.0%  avg_time=13.91s  (saved)
Running hindu_knowledge...
  accuracy=70.0%  avg_time=9.16s  (saved)
Running known_unknowns...
  accuracy=93.8%  avg_time=6.55s  (saved)
Running language_identification...
  accuracy=30.0%  avg_time=20.28s  (saved)
Running linguistics_puzzles...
  accuracy=0.0%  avg_time=36.09s  (saved)
Running logic_grid_puzzle...
  accuracy=40.0%  avg_time=38.36s  (saved)
Running logical_deduction...
  accuracy=75.0%  avg_time=15.53s  (saved)
Running misconceptions_russian...
  accuracy=5

,task,n_examples,accuracy,total_time_sec,avg_time_per_example_sec,model
0,auto_debugging,16,0.562500,324.28,20.27,Llama-3-8B
1,bbq_lite_json,20,0.150000,204.03,10.20,Llama-3-8B
2,code_line_description,16,1.000000,243.81,15.24,Llama-3-8B
3,conceptual_combinations,19,0.842105,280.83,14.78,Llama-3-8B
4,conlang_translation,20,0.100000,508.75,25.44,Llama-3-8B
5,emoji_movie,20,0.900000,202.40,10.12,Llama-3-8B
6,formal_fallacies_syllogisms_negation,20,0.500000,278.12,13.91,Llama-3-8B
7,hindu_knowledge,20,0.700000,183.13,9.16,Llama-3-8B
8,known_unknowns,16,0.937500,104.79,6.55,Llama-3-8B
9,language_identification,20,0.300000,405.54,20.28,Llama-3-8B


In [5]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils.bigbench_utils import run_task_eval

model_name = "Qwen/Qwen2.5-3B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cpu", torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

def ask_qwen(prompt):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=50, do_sample=False,
        temperature=None, top_p=None, top_k=None,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

def run_and_save(task_name, ask_fn, model_label, n=20):
    print(f"Running {task_name}...")
    result = run_task_eval(task_name, ask_fn, n=n)
    result["model"] = model_label
    df_new = pd.DataFrame([result])
    if os.path.exists(RESULTS_FILE):
        df_existing = pd.read_csv(RESULTS_FILE)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new
    df_combined.to_csv(RESULTS_FILE, index=False)
    print(f"  accuracy={result['accuracy']:.1%}  avg_time={result['avg_time_per_example_sec']}s  (saved)")
    return df_combined

RESULTS_FILE = "../results_qwen25_bbl.csv"

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [6]:

run_and_save("auto_debugging", ask_qwen, "Qwen2.5-3B")

run_and_save("bbq_lite_json", ask_qwen, "Qwen2.5-3B")

run_and_save("code_line_description", ask_qwen, "Qwen2.5-3B")

run_and_save("conceptual_combinations", ask_qwen, "Qwen2.5-3B")

run_and_save("conlang_translation", ask_qwen, "Qwen2.5-3B")

run_and_save("emoji_movie", ask_qwen, "Qwen2.5-3B")

run_and_save("formal_fallacies_syllogisms_negation", ask_qwen, "Qwen2.5-3B")

run_and_save("hindu_knowledge", ask_qwen, "Qwen2.5-3B")

run_and_save("known_unknowns", ask_qwen, "Qwen2.5-3B")

run_and_save("language_identification", ask_qwen, "Qwen2.5-3B")

run_and_save("linguistics_puzzles", ask_qwen, "Qwen2.5-3B")

run_and_save("logic_grid_puzzle", ask_qwen, "Qwen2.5-3B")

run_and_save("logical_deduction", ask_qwen, "Qwen2.5-3B")

run_and_save("misconceptions_russian", ask_qwen, "Qwen2.5-3B")

run_and_save("novel_concepts", ask_qwen, "Qwen2.5-3B")

run_and_save("operators", ask_qwen, "Qwen2.5-3B")

run_and_save("parsinlu_reading_comprehension", ask_qwen, "Qwen2.5-3B")

run_and_save("play_dialog_same_or_different", ask_qwen, "Qwen2.5-3B")

run_and_save("repeat_copy_logic", ask_qwen, "Qwen2.5-3B")

run_and_save("strange_stories", ask_qwen, "Qwen2.5-3B")

run_and_save("strategyqa", ask_qwen, "Qwen2.5-3B")

run_and_save("symbol_interpretation", ask_qwen, "Qwen2.5-3B")

run_and_save("vitaminc_fact_verification", ask_qwen, "Qwen2.5-3B")

run_and_save("winowhy", ask_qwen, "Qwen2.5-3B")

Running auto_debugging...
  accuracy=18.8%  avg_time=39.99s  (saved)
Running bbq_lite_json...
  accuracy=55.0%  avg_time=25.68s  (saved)
Running code_line_description...
  accuracy=87.5%  avg_time=35.24s  (saved)
Running conceptual_combinations...
  accuracy=78.9%  avg_time=34.74s  (saved)
Running conlang_translation...
  accuracy=20.0%  avg_time=82.08s  (saved)
Running emoji_movie...
  accuracy=60.0%  avg_time=24.94s  (saved)
Running formal_fallacies_syllogisms_negation...
  accuracy=75.0%  avg_time=31.16s  (saved)
Running hindu_knowledge...
  accuracy=65.0%  avg_time=22.41s  (saved)
Running known_unknowns...
  accuracy=81.2%  avg_time=17.91s  (saved)
Running language_identification...
  accuracy=20.0%  avg_time=47.2s  (saved)
Running linguistics_puzzles...
  accuracy=0.0%  avg_time=78.74s  (saved)
Running logic_grid_puzzle...
  accuracy=50.0%  avg_time=59.17s  (saved)
Running logical_deduction...
  accuracy=35.0%  avg_time=32.96s  (saved)
Running misconceptions_russian...
  accuracy=

,task,n_examples,accuracy,total_time_sec,avg_time_per_example_sec,model
0,auto_debugging,16,0.187500,639.78,39.99,Qwen2.5-3B
1,bbq_lite_json,20,0.550000,513.58,25.68,Qwen2.5-3B
2,code_line_description,16,0.875000,563.81,35.24,Qwen2.5-3B
3,conceptual_combinations,19,0.789474,660.07,34.74,Qwen2.5-3B
4,conlang_translation,20,0.200000,1641.64,82.08,Qwen2.5-3B
5,emoji_movie,20,0.600000,498.78,24.94,Qwen2.5-3B
6,formal_fallacies_syllogisms_negation,20,0.750000,623.20,31.16,Qwen2.5-3B
7,hindu_knowledge,20,0.650000,448.12,22.41,Qwen2.5-3B
8,known_unknowns,16,0.812500,286.51,17.91,Qwen2.5-3B
9,language_identification,20,0.200000,944.03,47.20,Qwen2.5-3B


In [ ]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils.bigbench_utils import run_task_eval

model_name = "ibm-granite/granite-4.0-micro"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cpu", torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

def ask_granite(prompt):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=50, do_sample=False,
        temperature=None, top_p=None, top_k=None,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

def run_and_save(task_name, ask_fn, model_label, n=20):
    print(f"Running {task_name}...")
    result = run_task_eval(task_name, ask_fn, n=n)
    result["model"] = model_label
    df_new = pd.DataFrame([result])
    if os.path.exists(RESULTS_FILE):
        df_existing = pd.read_csv(RESULTS_FILE)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new
    df_combined.to_csv(RESULTS_FILE, index=False)
    print(f"  accuracy={result['accuracy']:.1%}  avg_time={result['avg_time_per_example_sec']}s  (saved)")
    return df_combined

RESULTS_FILE = "../results_granite4_bbl.csv"

Loading weights:   0%|          | 0/322 [00:02<?, ?it/s]

In [8]:

run_and_save("auto_debugging", ask_granite, "Granite-4.0-Micro")

run_and_save("bbq_lite_json", ask_granite, "Granite-4.0-Micro")

run_and_save("code_line_description", ask_granite, "Granite-4.0-Micro")

run_and_save("conceptual_combinations", ask_granite, "Granite-4.0-Micro")

run_and_save("conlang_translation", ask_granite, "Granite-4.0-Micro")

run_and_save("emoji_movie", ask_granite, "Granite-4.0-Micro")

run_and_save("formal_fallacies_syllogisms_negation", ask_granite, "Granite-4.0-Micro")

run_and_save("hindu_knowledge", ask_granite, "Granite-4.0-Micro")

run_and_save("known_unknowns", ask_granite, "Granite-4.0-Micro")

run_and_save("language_identification", ask_granite, "Granite-4.0-Micro")

run_and_save("linguistics_puzzles", ask_granite, "Granite-4.0-Micro")

run_and_save("logic_grid_puzzle", ask_granite, "Granite-4.0-Micro")

run_and_save("logical_deduction", ask_granite, "Granite-4.0-Micro")

run_and_save("misconceptions_russian", ask_granite, "Granite-4.0-Micro")

run_and_save("novel_concepts", ask_granite, "Granite-4.0-Micro")

run_and_save("operators", ask_granite, "Granite-4.0-Micro")

run_and_save("parsinlu_reading_comprehension", ask_granite, "Granite-4.0-Micro")

run_and_save("play_dialog_same_or_different", ask_granite, "Granite-4.0-Micro")

run_and_save("repeat_copy_logic", ask_granite, "Granite-4.0-Micro")

run_and_save("strange_stories", ask_granite, "Granite-4.0-Micro")

run_and_save("strategyqa", ask_granite, "Granite-4.0-Micro")

run_and_save("symbol_interpretation", ask_granite, "Granite-4.0-Micro")

run_and_save("vitaminc_fact_verification", ask_granite, "Granite-4.0-Micro")

run_and_save("winowhy", ask_granite, "Granite-4.0-Micro")

Running auto_debugging...
  accuracy=43.8%  avg_time=34.21s  (saved)
Running bbq_lite_json...
  accuracy=45.0%  avg_time=17.34s  (saved)
Running code_line_description...
  accuracy=93.8%  avg_time=32.09s  (saved)
Running conceptual_combinations...
  accuracy=68.4%  avg_time=41.64s  (saved)
Running conlang_translation...
  accuracy=15.0%  avg_time=90.38s  (saved)
Running emoji_movie...
  accuracy=85.0%  avg_time=18.95s  (saved)
Running formal_fallacies_syllogisms_negation...
  accuracy=55.0%  avg_time=22.69s  (saved)
Running hindu_knowledge...
  accuracy=60.0%  avg_time=15.81s  (saved)
Running known_unknowns...
  accuracy=68.8%  avg_time=12.52s  (saved)
Running language_identification...
  accuracy=30.0%  avg_time=38.19s  (saved)
Running linguistics_puzzles...
  accuracy=0.0%  avg_time=43.74s  (saved)
Running logic_grid_puzzle...
  accuracy=50.0%  avg_time=51.71s  (saved)
Running logical_deduction...
  accuracy=65.0%  avg_time=29.58s  (saved)
Running misconceptions_russian...
  accuracy

,task,n_examples,accuracy,total_time_sec,avg_time_per_example_sec,model
0,auto_debugging,16,0.437500,547.34,34.21,Granite-4.0-Micro
1,bbq_lite_json,20,0.450000,346.86,17.34,Granite-4.0-Micro
2,code_line_description,16,0.937500,513.37,32.09,Granite-4.0-Micro
3,conceptual_combinations,19,0.684211,791.24,41.64,Granite-4.0-Micro
4,conlang_translation,20,0.150000,1807.50,90.38,Granite-4.0-Micro
5,emoji_movie,20,0.850000,379.04,18.95,Granite-4.0-Micro
6,formal_fallacies_syllogisms_negation,20,0.550000,453.76,22.69,Granite-4.0-Micro
7,hindu_knowledge,20,0.600000,316.23,15.81,Granite-4.0-Micro
8,known_unknowns,16,0.687500,200.26,12.52,Granite-4.0-Micro
9,language_identification,20,0.300000,763.76,38.19,Granite-4.0-Micro
